# Live Benchmark: Measured Token Usage with Claude & Gemini

This notebook makes **real API calls** and reports actual token counts returned by each model's
usage metadata. No estimates — everything is measured.

## Prerequisites

1. **NocturnusAI server running locally:**
   ```bash
   # In the repo root:
   ./gradlew :nocturnusai-server:run
   # or with Docker:
   docker run -p 9300:9300 ghcr.io/auctalis/nocturnusai:latest
   ```

2. **API keys** in environment:
   ```bash
   export ANTHROPIC_API_KEY=sk-ant-...
   export GOOGLE_API_KEY=AIza...
   ```

3. **Dependencies:**
   ```bash
   pip install -r ../requirements.txt
   ```

## Scenario

A **product support agent** handles a customer's issue over 15 turns.
We measure input token usage at each turn for two approaches:

- **Naive**: every turn sends the complete conversation history as context
- **NocturnusAI**: key facts are stored after each turn; only relevant facts are retrieved


In [ ]:
import os
import json
import time
import datetime
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

# ── Check environment ──────────────────────────────────────────────────────────
ANTHROPIC_KEY  = os.environ.get("ANTHROPIC_API_KEY")
GOOGLE_KEY     = os.environ.get("GOOGLE_API_KEY")
NOCTURNUS_URL  = os.environ.get("NOCTURNUS_URL", "http://localhost:9300")

print("Environment check:")
print(f"  ANTHROPIC_API_KEY : {'✅ set' if ANTHROPIC_KEY else '❌ missing'}")
print(f"  GOOGLE_API_KEY    : {'✅ set' if GOOGLE_KEY else '❌ missing'}")
print(f"  NOCTURNUS_URL     : {NOCTURNUS_URL}")

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
import urllib.request

try:
    with urllib.request.urlopen(f"{NOCTURNUS_URL}/health", timeout=3) as r:
        health = json.loads(r.read())
    print(f"✅ NocturnusAI server: {health.get('status', 'ok')}")
    NOCTURNUS_AVAILABLE = True
except Exception as e:
    print(f"⚠️  NocturnusAI server not reachable at {NOCTURNUS_URL}: {e}")
    print("    Start the server and re-run this cell. Token savings section will be skipped.")
    NOCTURNUS_AVAILABLE = False

## The conversation scenario

A 15-turn support conversation. Each tuple is `(user_message, key_facts_to_store)`.
The facts list shows what NocturnusAI extracts and stores after each turn.


In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful product support agent for NocturnusAI, a logic-based inference engine. "
    "Be concise. Ask clarifying questions when needed."
)

# Each turn: (user_message, [(predicate, [args], ...]) — facts extracted for NocturnusAI)
TURNS = [
    (
        "Hi, I'm having trouble with the API. My requests are timing out after about 30 seconds.",
        [("issue_type", ["user", "api_timeout"]), ("timeout_threshold", ["user", "30s"])]
    ),
    (
        "I'm on the Pro plan. We have about 500 concurrent users.",
        [("account_plan", ["user", "pro"]), ("concurrent_users", ["user", "500"])]
    ),
    (
        "We're using the Python SDK, version 0.3.8.",
        [("sdk_language", ["user", "python"]), ("sdk_version", ["user", "0.3.8"])]
    ),
    (
        "The errors started happening after we scaled to 500 users, around two days ago.",
        [("issue_started", ["user", "2_days_ago"]), ("scale_trigger", ["user", "500_users"])]
    ),
    (
        "We're running on AWS EC2, t3.medium instances behind an ALB.",
        [("infra_provider", ["user", "aws"]), ("instance_type", ["user", "t3_medium"]), ("load_balancer", ["user", "alb"])]
    ),
    (
        "The ALB timeout is set to 60 seconds. Should I increase it?",
        [("alb_timeout_config", ["user", "60s"]), ("user_question", ["user", "increase_alb_timeout"])]
    ),
    (
        "I checked the logs and I see: 'Connection pool exhausted after 30000ms'. About 5% of requests.",
        [("error_message", ["user", "connection_pool_exhausted"]), ("error_rate_pct", ["user", "5"]), ("root_cause_candidate", ["user", "pool_exhaustion"])]
    ),
    (
        "We have max_connections=10 in our NocturnusAI client config.",
        [("client_max_connections", ["user", "10"]), ("config_issue_candidate", ["user", "low_pool_size"])]
    ),
    (
        "How many connections should we set for 500 concurrent users?",
        [("user_question", ["user", "recommended_pool_size_for_500_users"])]
    ),
    (
        "I updated max_connections to 100. Still getting some timeouts but much less frequent now.",
        [("config_change", ["user", "max_connections_100"]), ("partial_improvement", ["user", "true"])]
    ),
    (
        "The remaining timeouts seem to be on the /infer endpoint specifically.",
        [("affected_endpoint", ["user", "infer"]), ("issue_narrowed", ["user", "true"])]
    ),
    (
        "Our inference rules are quite complex — some have 8-10 body atoms with variables.",
        [("rule_complexity", ["user", "high"]), ("body_atoms_count", ["user", "8_to_10"]), ("uses_variables", ["user", "true"])]
    ),
    (
        "We have about 50,000 facts in the knowledge base.",
        [("knowledge_base_size", ["user", "50000_facts"])]
    ),
    (
        "How can I set a per-query timeout to avoid blocking the pool on slow inferences?",
        [("user_question", ["user", "per_query_timeout"])]
    ),
    (
        "Can you summarize the issue and the fixes we've applied so far?",
        []
    ),
]

print(f"Scenario: {len(TURNS)} turns, product support conversation")
print(f"System prompt: {len(SYSTEM_PROMPT.split())} words")

## Part 1: Naive approach — full context replay

Every turn sends the entire conversation history. Token count grows with every exchange.


In [ ]:
import anthropic

naive_claude_tokens = []

if ANTHROPIC_KEY:
    claude = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
    history = []   # grows with each turn

    for i, (user_msg, _) in enumerate(TURNS):
        history.append({"role": "user", "content": user_msg})

        # Full history sent every turn
        response = claude.messages.create(
            model="claude-opus-4-6",
            max_tokens=200,
            system=SYSTEM_PROMPT,
            messages=history,
        )
        assistant_reply = response.content[0].text
        history.append({"role": "assistant", "content": assistant_reply})

        input_tokens = response.usage.input_tokens
        naive_claude_tokens.append(input_tokens)
        print(f"  Turn {i+1:2d}: {input_tokens:,} input tokens")
        time.sleep(0.5)  # rate limit courtesy

    total = sum(naive_claude_tokens)
    print(f"\nTotal input tokens (naive, Claude): {total:,}")
    print(f"Average per turn:                   {total//len(TURNS):,}")
else:
    print("Skipping — ANTHROPIC_API_KEY not set")
    # Fallback: simulate with token counter
    naive_claude_tokens = []

In [ ]:
import google.generativeai as genai

naive_gemini_tokens = []

if GOOGLE_KEY:
    genai.configure(api_key=GOOGLE_KEY)
    gemini = genai.GenerativeModel(
        "gemini-1.5-flash",
        system_instruction=SYSTEM_PROMPT,
    )
    history_text = ""   # naive: concatenate all history as one string

    for i, (user_msg, _) in enumerate(TURNS):
        prompt = history_text + f"\nUser: {user_msg}\nAssistant:"

        # Count tokens before sending (Gemini provides this)
        token_count_resp = gemini.count_tokens(prompt)
        input_tokens = token_count_resp.total_tokens

        response = gemini.generate_content(prompt)
        assistant_reply = response.text

        history_text += f"\nUser: {user_msg}\nAssistant: {assistant_reply}"
        naive_gemini_tokens.append(input_tokens)
        print(f"  Turn {i+1:2d}: {input_tokens:,} input tokens")
        time.sleep(0.3)

    total = sum(naive_gemini_tokens)
    print(f"\nTotal input tokens (naive, Gemini): {total:,}")
    print(f"Average per turn:                   {total//len(TURNS):,}")
else:
    print("Skipping — GOOGLE_API_KEY not set")
    naive_gemini_tokens = []

## Part 2: NocturnusAI approach — structured fact retrieval

After each turn, key facts are extracted and stored. The next turn retrieves only relevant facts
— a tiny, flat context instead of growing history.


In [ ]:
if NOCTURNUS_AVAILABLE:
    from nocturnusai import SyncNocturnusAIClient

    # Unique tenant for this benchmark run
    TENANT = f"bench_{datetime.datetime.now().strftime('%H%M%S')}"
    noct = SyncNocturnusAIClient(NOCTURNUS_URL, tenant_id=TENANT)

    # Bootstrap tenant
    import urllib.request, urllib.error
    req = urllib.request.Request(
        f"{NOCTURNUS_URL}/admin/databases/default/tenants",
        data=json.dumps({"tenantId": TENANT}).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req) as r:
        pass
    print(f"✅ NocturnusAI tenant created: {TENANT}")
else:
    print("Skipping NocturnusAI setup — server not available")

In [ ]:
noct_claude_tokens = []

if NOCTURNUS_AVAILABLE and ANTHROPIC_KEY:
    claude = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

    for i, (user_msg, facts_to_store) in enumerate(TURNS):
        # Store facts from this turn
        for predicate, args in facts_to_store:
            try:
                noct.assert_fact(predicate, args)
            except Exception:
                pass  # skip duplicates

        # Retrieve all stored facts as compact context
        all_facts = noct.query("?pred", ["user", "?val"])
        facts_context = "Known facts about this user:\n"
        for fact in all_facts[:15]:  # cap at 15 most relevant
            facts_context += f"  - {fact.predicate}({', '.join(fact.args)})\n"

        # Send only: system + facts summary + current message (no history)
        messages = [
            {"role": "user", "content": f"{facts_context}\nUser message: {user_msg}"}
        ]

        response = claude.messages.create(
            model="claude-opus-4-6",
            max_tokens=200,
            system=SYSTEM_PROMPT,
            messages=messages,
        )

        input_tokens = response.usage.input_tokens
        noct_claude_tokens.append(input_tokens)
        print(f"  Turn {i+1:2d}: {input_tokens:,} input tokens  (facts stored: {len(facts_to_store)})")
        time.sleep(0.5)

    total = sum(noct_claude_tokens)
    print(f"\nTotal input tokens (NocturnusAI, Claude): {total:,}")
    print(f"Average per turn:                         {total//len(TURNS):,}")
elif not NOCTURNUS_AVAILABLE:
    print("Skipping — NocturnusAI server not available")
else:
    print("Skipping — ANTHROPIC_API_KEY not set")
    noct_claude_tokens = []

In [ ]:
noct_gemini_tokens = []

if NOCTURNUS_AVAILABLE and GOOGLE_KEY:
    genai.configure(api_key=GOOGLE_KEY)
    gemini = genai.GenerativeModel("gemini-1.5-flash", system_instruction=SYSTEM_PROMPT)

    # Use same stored facts from the Claude run (or re-query)
    for i, (user_msg, _) in enumerate(TURNS):
        # Retrieve facts from NocturnusAI
        all_facts = noct.query("?pred", ["user", "?val"])
        facts_context = "Known facts about this user:\n"
        for fact in all_facts[:15]:
            facts_context += f"  - {fact.predicate}({', '.join(fact.args)})\n"

        prompt = f"{facts_context}\nUser message: {user_msg}\nAssistant:"

        token_count_resp = gemini.count_tokens(prompt)
        input_tokens = token_count_resp.total_tokens

        gemini.generate_content(prompt)
        noct_gemini_tokens.append(input_tokens)
        print(f"  Turn {i+1:2d}: {input_tokens:,} input tokens")
        time.sleep(0.3)

    total = sum(noct_gemini_tokens)
    print(f"\nTotal input tokens (NocturnusAI, Gemini): {total:,}")
    print(f"Average per turn:                         {total//len(TURNS):,}")
elif not NOCTURNUS_AVAILABLE:
    print("Skipping — NocturnusAI server not available")
else:
    print("Skipping — GOOGLE_API_KEY not set")
    noct_gemini_tokens = []

## Results: measured token usage


In [ ]:
turns = list(range(1, len(TURNS) + 1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (model_name, naive_tok, noct_tok, color) in zip(axes, [
    ("Claude Opus 4", naive_claude_tokens, noct_claude_tokens, "#ff6b35"),
    ("Gemini 1.5 Flash", naive_gemini_tokens, noct_gemini_tokens, "#4285f4"),
]):
    if not naive_tok or not noct_tok:
        ax.text(0.5, 0.5, "No data\n(API key not set or\nserver not running)",
                ha='center', va='center', transform=ax.transAxes, fontsize=12, color='gray')
        ax.set_title(model_name)
        continue

    ax.plot(turns, naive_tok, color='#e55', linewidth=2, marker='o', markersize=4,
            label='Naive (full history)')
    ax.plot(turns, noct_tok, color=color, linewidth=2, marker='s', markersize=4,
            label='NocturnusAI (retrieved facts)')
    ax.fill_between(turns, noct_tok, naive_tok, alpha=0.1, color='#e55')

    total_naive = sum(naive_tok)
    total_noct  = sum(noct_tok)
    ratio = total_naive / total_noct if total_noct > 0 else 0
    ax.set_title(f"{model_name}: {ratio:.1f}× fewer tokens", fontweight='bold')
    ax.set_xlabel('Conversation turn')
    ax.set_ylabel('Input tokens')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    ax.legend(frameon=False, fontsize=9)

plt.suptitle('Measured token usage per turn: Naive vs NocturnusAI', fontsize=13)
plt.tight_layout()
plt.savefig('../results/04_live_token_usage.png', bbox_inches='tight')
plt.show()
print("Saved → results/04_live_token_usage.png")

In [ ]:
# Extrapolate measured averages to production scale
SCALE = 50_000   # turns/month for the extrapolation

CLAUDE_OPUS_INPUT_PER_1M  = 15.00
GEMINI_FLASH_INPUT_PER_1M =  0.075

print(f"Extrapolation: {SCALE:,} turns/month at production scale\n")
print(f"{'Model':<20} {'Approach':<25} {'Avg tokens/turn':>18} {'Monthly cost':>15}")
print("-" * 82)

for model_name, naive_tok, noct_tok, price in [
    ("Claude Opus 4",    naive_claude_tokens,  noct_claude_tokens,  CLAUDE_OPUS_INPUT_PER_1M),
    ("Gemini 1.5 Flash", naive_gemini_tokens,  noct_gemini_tokens,  GEMINI_FLASH_INPUT_PER_1M),
]:
    if not naive_tok:
        print(f"{model_name:<20} (no data)")
        continue

    avg_naive = sum(naive_tok) / len(naive_tok)
    avg_noct  = sum(noct_tok)  / len(noct_tok) if noct_tok else None

    cost_naive = SCALE * avg_naive * price / 1_000_000
    print(f"{model_name:<20} {'Naive':25} {avg_naive:>18,.0f} ${cost_naive:>14,.2f}")

    if avg_noct:
        cost_noct = SCALE * avg_noct * price / 1_000_000
        savings_ratio = avg_naive / avg_noct
        print(f"{'':<20} {'NocturnusAI':25} {avg_noct:>18,.0f} ${cost_noct:>14,.2f}  ({savings_ratio:.1f}× cheaper)")
    print()

# Save results as JSON for community comparison
results = {
    "timestamp": datetime.datetime.now().isoformat(),
    "scenario": "product_support_15_turns",
    "turns": len(TURNS),
    "naive_claude_tokens": naive_claude_tokens,
    "noct_claude_tokens":  noct_claude_tokens,
    "naive_gemini_tokens": naive_gemini_tokens,
    "noct_gemini_tokens":  noct_gemini_tokens,
}
with open('../results/benchmark_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\nResults saved → results/benchmark_results.json")

## Share your results!

Ran this on a different conversation type or with different facts? Open a PR:

1. Copy your `results/benchmark_results.json`
2. Add a description of your scenario
3. Open a PR to `nocturnusai-bench/community/`

See [`CONTRIBUTING.md`](../CONTRIBUTING.md) for details.
